# 📚 RAG en vivo: pregúntale a El Principito

En este notebook vamos a construir un sistema RAG paso a paso que permite hacerle preguntas al libro *El Principito* de Antoine de Saint-Exupéry.

**Lo que vamos a hacer:**
1. Instalar dependencias
2. Cargar el PDF
3. Dividir en chunks
4. Crear embeddings y guardar en ChromaDB
5. Hacer preguntas en vivo

> ⚠️ **Antes de empezar:** necesitas una API key de OpenRouter (gratis en [openrouter.ai](https://openrouter.ai))

## 0. Instalación de dependencias

Solo necesitas correr esta celda una vez.

In [ ]:
%pip install -r requirements.txt

## 1. Configuración

Ingresa tu API key de OpenRouter. Puedes obtenerla gratis en [openrouter.ai/keys](https://openrouter.ai/keys).

In [1]:
import os
from getpass import getpass

# Ingresa tu API key (no se muestra en pantalla)
os.environ["OPENROUTER_API_KEY"] = getpass("🔑 OpenRouter API Key: ")

# Ruta al PDF — asegúrate de que el archivo esté en la misma carpeta que este notebook
PDF_PATH = "el_principito.pdf"

print("✅ Configuración lista")

✅ Configuración lista


## 2. Cargar el PDF

Cargamos el PDF usando `PyPDFLoader`. Cada página del PDF se convierte en un objeto `Document` con su texto y metadatos (número de página, nombre del archivo).

In [2]:
import re
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

# Limpiar espacios y tabs extra que genera la extracción del PDF
for doc in docs:
    doc.page_content = re.sub(r'[ \t]+', ' ', doc.page_content)  # colapsa espacios/tabs
    doc.page_content = re.sub(r'\n{3,}', '\n\n', doc.page_content)  # colapsa saltos extra
    doc.page_content = doc.page_content.strip()

print(f"📄 Páginas cargadas: {len(docs)}")
print(f"\n--- Ejemplo: quinta página ---")
print(docs[5].page_content[:500])

c:\Users\lcgallardov\Documents\PCD\clase_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📄 Páginas cargadas: 47

--- Ejemplo: quinta página ---
—Porque en mi tierra es todo tan pequeño…
Se inclinó hacia el dibujo y exclamó:
—¡Bueno, no tan pequeño…! Está dormido…
Y así fue como conocí al principito.
 
 
III
 
Me costó mucho tiempo comprender de dónde venía. El principito, que me
hacía muchas preguntas, jamás parecía oír las mías. Fueron palabras
pronunciadas al azar, las que poco a poco me revelaron todo. Así, cuando
distinguió por vez primera mi avión (no dibujaré mi avión, por tratarse de un
dibujo demasiado complicado para mí) me pre


## 3. Dividir en chunks

El PDF completo es demasiado largo para enviarlo al LLM de una sola vez. Lo dividimos en fragmentos pequeños (*chunks*) con un poco de solapamiento para no perder contexto entre uno y otro.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # caracteres por chunk
    chunk_overlap=50,    # solapamiento entre chunks
)
chunks = splitter.split_documents(docs)

print(f"✂️  Total de chunks: {len(chunks)}")
print(f"\n--- Ejemplo: chunk #10 ---")
print(chunks[10].page_content)
print(f"\nMetadatos: {chunks[10].metadata}")

✂️  Total de chunks: 193

--- Ejemplo: chunk #10 ---
del lugar habitado más próximo. Estaba más aislado que un náufrago en una
balsa en medio del océano. Imagínense, pues, mi sorpresa cuando al amanecer
me despertó una extraña vocecita que decía:
— ¡Por favor... píntame un cordero!
—¿Eh?
—¡Píntame un cordero!
Me puse en pie de un salto como herido por el rayo. Me froté los ojos.
Miré a mi alrededor. Vi a un extraordinario muchachito que me miraba
gravemente. Ahí tienen el mejor retrato que más tarde logré hacer de él,

Metadatos: {'producer': 'calibre 2.53.0 [http://calibre-ebook.com]', 'creator': 'calibre 2.53.0 [http://calibre-ebook.com]', 'creationdate': '2018-02-27T10:18:33+00:00', 'author': 'Antoine De Saint-Exupéry', 'title': 'El Principito', 'source': 'el_principito.pdf', 'total_pages': 47, 'page': 3, 'page_label': '4'}


## 4. Crear embeddings y guardar en ChromaDB

Convertimos cada chunk a un vector numérico usando un modelo de embeddings. Estos vectores se guardan en ChromaDB, una base de datos vectorial local.

> ⏳ Esta celda puede tardar 1-2 minutos la primera vez — está descargando el modelo de embeddings.

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("⏳ Creando embeddings... (puede tardar un momento)")

embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",  # multilingüe, funciona bien en español
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_principito",  # se guarda localmente
)

print(f"✅ ChromaDB listo con {vectorstore._collection.count()} vectores")

⏳ Creando embeddings... (puede tardar un momento)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5850.21it/s]


✅ ChromaDB listo con 193 vectores


## 5. Crear el retriever y el LLM

El retriever busca los chunks más similares a una pregunta. El LLM recibe esos chunks como contexto y genera la respuesta.

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Retriever: recupera los 3 chunks más relevantes
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

# LLM via OpenRouter
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    model="openai/gpt-oss-20b:free",
    temperature=0.0,
)

# Prompt: instruye al LLM a responder SOLO con el contexto
prompt = PromptTemplate.from_template("""
Eres un asistente experto en el libro El Principito de Antoine de Saint-Exupéry.
Responde la pregunta usando SOLO la siguiente información del libro.
Si la respuesta no está en el contexto, di "No encontré esa información en el texto."

Contexto:
{context}

Pregunta: {question}

Respuesta:
""")

# Cadena LCEL
chain = (
    {"question": RunnablePassthrough(),
     "context": retriever}
    | prompt
    | llm
)

print("✅ Pipeline RAG listo")

✅ Pipeline RAG listo


## 6. ¡A preguntar! ✅ Preguntas que funcionan bien

Estas preguntas tienen respuesta clara en el texto.

In [7]:
def preguntar(pregunta):
    """Hace una pregunta al pipeline RAG y muestra la respuesta con los chunks recuperados."""
    print(f"❓ Pregunta: {pregunta}")
    print("-" * 60)
    
    # Ver qué chunks recuperó el retriever
    chunks_recuperados = retriever.invoke(pregunta)
    print("📎 Chunks recuperados:")
    for i, chunk in enumerate(chunks_recuperados, 1):
        print(f"  [{i}] Página {chunk.metadata.get('page', '?')}: {chunk.page_content[:120]}...")
    
    print()
    respuesta = chain.invoke(pregunta)
    print(f"🤖 Respuesta:\n{respuesta.content}")
    print("=" * 60)

In [8]:
# ✅ Pregunta directa con respuesta clara en el texto
preguntar("¿Qué le pidió el principito al piloto que dibujara?")

❓ Pregunta: ¿Qué le pidió el principito al piloto que dibujara?
------------------------------------------------------------
📎 Chunks recuperados:
  [1] Página 5: dibujo demasiado complicado para mí) me preguntó:
—¿Qué cosa es esa? —Eso no es una cosa. Eso vuela. Es un avión, mi
avi...
  [2] Página 5: —Porque en mi tierra es todo tan pequeño…
Se inclinó hacia el dibujo y exclamó:
—¡Bueno, no tan pequeño…! Está dormido…
...
  [3] Página 39: que nuevamente se había sentado junto a mí.
—¿Qué promesa?
—Ya sabes... el bozal para mi cordero... soy responsable de m...

🤖 Respuesta:
El principito le pidió al piloto que dibujara una oveja.


In [9]:
# ✅ Pregunta semántica — no usa las palabras exactas del libro
preguntar("¿Qué significa domesticar según el zorro?")

❓ Pregunta: ¿Qué significa domesticar según el zorro?
------------------------------------------------------------
📎 Chunks recuperados:
  [1] Página 32: —Estoy aquí, bajo el manzano —dijo la voz.
—¿Quién eres tú? —preguntó el principito—. ¡Qué bonito eres!
—Soy un zorro —d...
  [2] Página 32: "domesticar"?
—Los hombres —dijo el zorro— tienen escopetas y cazan. ¡Es muy
molesto! Pero también crían gallinas. Es lo...
  [3] Página 32: que un muchachito igual a otros cien mil muchachitos y no te necesito para
nada. Tampoco tú tienes necesidad de mí y no ...

🤖 Respuesta:
Domesticar, según el zorro, significa crear vínculos que hacen que ambos se necesiten mutuamente; al domesticar al otro, cada uno se vuelve único y esencial en el mundo del otro.


In [10]:
# ✅ Pregunta de comprensión
preguntar("¿Por qué el principito abandonó su planeta?")

❓ Pregunta: ¿Por qué el principito abandonó su planeta?
------------------------------------------------------------
📎 Chunks recuperados:
  [1] Página 23: El principito abandonó aquel planeta.
"Las personas mayores, decididamente, son extraordinarias", se decía a sí
mismo co...
  [2] Página 6: Entonces el principito señaló con gravedad:
—¡No importa, es tan pequeña mi tierra!
Y agregó, quizás, con un poco de mel...
  [3] Página 19: el principito durante su viaje.
 
 
XII
 
El tercer planeta estaba habitado por un bebedor. Fue una visita muy corta,...

🤖 Respuesta:
No encontré esa información en el texto.


## 7. ❌ Pregunta que el RAG NO puede responder

Una de las ventajas del RAG es que **sabe cuándo no sabe** — si el contexto no tiene la respuesta, debería decirlo.

In [11]:
# ❌ Pregunta fuera del libro — el modelo debería responder que no tiene esa info
preguntar("¿En qué año nació Antoine de Saint-Exupéry?")

❓ Pregunta: ¿En qué año nació Antoine de Saint-Exupéry?
------------------------------------------------------------
📎 Chunks recuperados:
  [1] Página 0: El Principito
 
Por
 
Antoine De Saint-Exupéry...
  [2] Página 23: "Este hombre, quizás, es absurdo. Sin embargo, es menos absurdo que el
rey, el vanidoso, el hombre de negocios y el bebe...
  [3] Página 27: —De las flores no tomamos nota.
—¿Por qué? ¡Son lo más bonito!
—Porque las flores son efímeras.
—¿Qué significa "efímera...

🤖 Respuesta:
No encontré esa información en el texto.


In [12]:
# ❌ Pregunta ambigua que puede confundir al retriever
preguntar("¿Qué pasó al final?")

❓ Pregunta: ¿Qué pasó al final?
------------------------------------------------------------
📎 Chunks recuperados:
  [1] Página 12: salir de allí una aparición milagrosa; pero la flor no acababa de preparar su...
  [2] Página 41: muere a tiros de carabina.
—Me alegra —dijo el principito— que hayas encontrado lo que faltaba a
tu máquina. Así podrás ...
  [3] Página 44: Me senté, ya no podía mantenerme en pie.
—Ahí está... eso es todo...
Vaciló todavía un instante, luego se levantó y dio ...

🤖 Respuesta:
No encontré esa información en el texto.


## 8. 🔍 Pregunta libre

¡Escribe tu propia pregunta!

In [ ]:
mi_pregunta = input("✏️  Escribe tu pregunta: ")
preguntar(mi_pregunta)

---

## 💡 ¿Qué aprendimos?

| Concepto | Lo que vimos |
|---|---|
| **Document Loader** | `PyPDFLoader` convierte el PDF en objetos `Document` |
| **Text Splitter** | Dividimos en chunks de 500 caracteres con solapamiento |
| **Embeddings** | `multilingual-e5-base` convierte texto en vectores numéricos |
| **Vector Store** | ChromaDB almacena y busca los vectores localmente |
| **Retriever** | Recupera los 3 chunks más similares a la pregunta |
| **LLM** | Llama 3.3 70B genera la respuesta usando solo el contexto |
| **LCEL** | `|` encadena todos los componentes en un pipeline limpio |

> El mismo código funciona con cualquier PDF — solo cambia `PDF_PATH` por el documento que necesites.